### Broadcast Variables & Accumulators in Spark DataFrame

#### Broadcast vs Accumulator — Summary
![image_1775548856760.png](./image_1775548856760.png "image_1775548856760.png")

#### PART 1 — Broadcast Variables

What is it?
A read-only lookup table sent once to every executor and **cached** there for the lifetime of the application. Without broadcast, Spark sends the variable with every single task — very inefficient.

![image_1775459194332.png](./image_1775459194332.png "image_1775459194332.png")

##### Example 1 — Country Code Lookup

Sample Data Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, DoubleType


data = [
    (1, "Alice",   "IN", "Electronics", 5000),
    (2, "Bob",     "US", "Clothing",    3000),
    (3, "Charlie", "IN", "Electronics", 7000),
    (4, "Diana",   "EU", "Clothing",    4000),
    (5, "Eve",     "US", "Electronics", 8000),
    (6, "Frank",   "IN", "Food",        1500),   # unknown category
    (7, "Grace",   "XX", "Electronics", 9000),   # unknown country
]
df = spark.createDataFrame(data, ["order_id", "name", "country", "category", "amount"])

In [0]:
# Lookup table — too large to hardcode in every task efficiently
country_map = {
    "IN": "India",
    "US": "United States",
    "EU": "European Union"
}

# Broadcast it once to all executors 
# broadcase is not available in serverless computing. hence directly accessing the map.
bc_country = spark.sparkContext.broadcast(country_map)

# Use inside a UDF — bc_country.value accesses the dict on executor
country_lookup = udf(
    lambda code: bc_country.value.get(code, "Unknown"),
    StringType()
)

df.withColumn("country_name", country_lookup(col("country"))).show()



##### Example 2 — Category Discount Lookup

In [0]:
disc_map = {
    "Electronics": 10,
    "Clothing": 5,
    "Food": 2
}

bc_disc = spark.sparkContext.broadcast(disc_map)

discount_lookup = udf(
    lambda cat: bc_disc.value.get(cat, 0), DoubleType()
)
df.withColumn("discount", df.amount * discount_lookup(col("category")) / 100) \
  .withColumn("Rate After Discount", col("amount") -  col("discount")).show()

##### Broadcast Join (without UDF)

In [0]:
from pyspark.sql.functions import broadcast
disc_df = spark.createDataFrame([
    ("Electronics", 5),
    ("Clothing", 10),
    ("Food", 2)],
    ["category", "discount_pct"])

df.join(broadcast(disc_df), "category", "left") \
  .withColumn("discount Amount", col("amount") * col("discount_pct")/100) \
  .withColumn("Rate After Discount", col("amount") -  col("discount Amount")).show()

 

#### PART 2 — Accumulators

What is it?
A **write-only** variable on executors that only the driver can read. Used to count events or aggregate metrics across tasks without collecting all data to the driver.

![image_1775469536451.png](./image_1775469536451.png "image_1775469536451.png")

##### Example 1 — Count Bad Records


In [0]:
# Define accumulators
unknown_country_count = spark.sparkContext.accumulator(0)
unknown_category_count = spark.sparkContext.accumulator(0)

known_countries  = {"IN", "US", "EU"}
known_categories = {"Electronics", "Clothing", "Food"}

def validate_and_tag(row):
    global unknown_country_count, unknown_category_count

    if row["country"] not in known_countries:
        unknown_country_count += 1         # executor writes

    if row["category"] not in known_categories:
        unknown_category_count += 1        # executor writes

# Use foreach (action) — guarantees exactly-once execution
df.foreach(validate_and_tag)

# Driver reads the final values
print(f"Unknown countries  : {unknown_country_count.value}")
print(f"Unknown categories : {unknown_category_count.value}")

##### Example 2 — Multiple Accumulators for Data Quality

In [0]:
null_amount_count  = spark.sparkContext.accumulator(0)
high_value_count   = spark.sparkContext.accumulator(0)
total_rows         = spark.sparkContext.accumulator(0)

def audit_row(row):
    total_rows.add(1)

    if row["amount"] is None:
        null_amount_count.add(1)

    if row["amount"] and row["amount"] > 6000:
        high_value_count.add(1)

df.foreach(audit_row)

print(f"Total rows processed : {total_rows.value}")
print(f"Null amount rows     : {null_amount_count.value}")
print(f"High value orders    : {high_value_count.value}")

##### ⚠️ Critical Gotcha — Accumulators in Transformations 

In [0]:
bad_count = spark.sparkContext.accumulator(0)

# ❌ WRONG — inside a transformation (lazy), may be double-counted
df2 = df.filter(lambda row: (bad_count.add(1) or True))  # unreliable!
df2.count()   # may run filter twice due to recomputation

# ✅ CORRECT — inside an action (foreach), executes exactly once
df.foreach(lambda row: bad_count.add(1))

> Rule: Only use accumulators inside actions (foreach, foreachPartition), never inside transformations.


##### global Declaration for Accumulators — Is It Necessary?

The difference comes down to how Python handles variable access — it depends on whether you are reassigning the variable or calling a method on it.

Why Example 1 Uses global

`unknown_country_count += 1`

+= is syntactic sugar for reassignment:

`unknown_country_count = unknown_country_count + 1`

Python sees this as creating a new local variable inside the function. Without global, Python throws:

`UnboundLocalError: local variable 'unknown_country_count' referenced before assignment`


**So global is required here to tell Python — "use the outer variable, don't create a new local one.
**

Why Example 2 Does NOT Need global

`null_amount_count.add(1)
total_rows.add(1)`

.add() is a **method call on an existing object** — Python is not reassigning the variable, it's just accessing the object and calling its method. Python can find it in the enclosing scope without global.


![image_1775550020859.png](./image_1775550020859.png "image_1775550020859.png")

Recommendation — Always Use .add() to Avoid global

![image_1775550217985.png](./image_1775550217985.png "image_1775550217985.png")

.add() is the idiomatic and safer way to update accumulators in PySpark — it avoids global declarations and is consistent across all accumulator types (int, float, custom).

#### Broadcast Variable & DataFrame Size Limits

##### Broadcast Variable Maximum Size - Hard Limit

spark.driver.maxResultSize   (default: 1GB)

##### Soft Limit


The broadcast variable must fit in driver memory first (driver serializes and sends it), then in executor memory.

Practical Soft Limit — spark.sql.autoBroadcastJoinThreshold

In [0]:
# Default: 10 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10MB

# Increase if you have more memory
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024) # 100MB

# Disable auto broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

This threshold applies to automatic broadcast join decisions by Catalyst optimizer.



##### Manual vs Automatic Broadcast

![image_1775578097927.png](./image_1775578097927.png "image_1775578097927.png")

In [0]:
from pyspark.sql.functions import broadcast

# Auto — Catalyst broadcasts if disc_df < 10MB
df.join(disc_df, "id")

# Manual — you force broadcast regardless of size
df.join(broadcast(disc_df), "id")   # you take responsibility for memory

##### What Determines the Real Limit?


![image_1775578234454.png](./image_1775578234454.png "image_1775578234454.png")

##### Recommended Size Guidelines
![image_1775578413259.png](./image_1775578413259.png "image_1775578413259.png")

##### How to Check DataFrame Size Before Broadcasting

In [0]:
# Rough size estimate in bytes
from pyspark.sql.functions import sum, length, col

# Option 1 — check plan statistics
df.explain(True)   # look for "sizeInBytes" in optimized plan

# Option 2 — cache and check Storage UI
df.cache()
df.count()   # trigger caching
# → check Spark UI → Storage tab → size in memory

# Option 3 — config check
#print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

##### Summary

![image_1775579727938.png](./image_1775579727938.png "image_1775579727938.png")